# Example: Building the Cobb-Douglas Allocator

In this example, we generate a synthetic market path, compute SIM parameters and the EMA crossover sentiment signal, then use **Cobb-Douglas utility maximization** to allocate capital across a 5-asset universe. We visualize how preferences shift with sentiment and compare the Cobb-Douglas allocation to two alternatives: CES and log-linear utility.

> **By the end of this example, you will be able to:**
> * Generate synthetic per-ticker prices via SIM and compute the lambda sentiment signal
> * Solve the Cobb-Douglas budget-constrained allocation problem analytically
> * Compare allocation behavior across three utility functions

## Setup, Data and Prerequisites

In [ ]:
include("Include.jl");

Define the constants we'll use throughout the notebook — EMA windows, gain, budget, and time parameters.

In [ ]:
let
    # Constants -
    global Δt = 1.0 / 252.0;       # years per trading day
    global N_short = 21;             # short EMA window (~1 month)
    global N_long = 63;              # long EMA window (~1 quarter)
    global N_growth = 10;            # market growth EMA window
    global GAIN = 10.0;              # lambda sensitivity
    global B₀ = 10000.0;            # initial budget ($)
    global offset = N_short + N_long; # warmup period (84 days)
    global n_trading_days = 252;      # 1 year of trading after warmup

    # Asset universe -
    global tickers = ["LargeCap", "SmallCap", "International", "Bond", "Commodity"];

    # SIM parameters (αᵢ, βᵢ, σᵢ) — estimated from "historical" data -
    global sim_params = Dict(
        "LargeCap"      => (0.0002, 1.10, 0.010),
        "SmallCap"      => (0.0003, 1.35, 0.014),
        "International" => (0.0001, 0.95, 0.012),
        "Bond"          => (0.0001, -0.15, 0.003),
        "Commodity"     => (0.0001, 0.60, 0.013)
    );

    println("Constants loaded: offset = $(offset) days, trading period = $(n_trading_days) days")
    println("Tickers: $(tickers)")
end

___
## Task 1: Generate Synthetic Market Data and Compute Sentiment Signal
We generate a synthetic market index using geometric Brownian motion, compute per-ticker prices via the Single Index Model, and build the EMA crossover sentiment signal $\lambda_t$ that will feed into the preference weights.

> **What should you see?** The market index follows a random walk. The short EMA hugs price; the long EMA smooths it. When short crosses below long, $\lambda > 0$ (bearish); when above, $\lambda < 0$ (bullish). Per-ticker paths are correlated with the market through their $\beta_i$.

In [ ]:
let
    # Total days needed: warmup + trading period -
    T_total = offset + n_trading_days;
    K = length(tickers);

    # Generate synthetic market index (GBM: μ = 8% annual, σ = 18% annual) -
    μ_market = 0.08;
    σ_market = 0.18;
    S₀ = 100.0;

    global market_prices = zeros(T_total);
    market_prices[1] = S₀;
    for t in 2:T_total
        z = randn();
        market_prices[t] = market_prices[t-1] * exp((μ_market - 0.5 * σ_market^2) * Δt + σ_market * sqrt(Δt) * z);
    end

    # Compute EMAs and lambda -
    global ema_short = compute_ema(market_prices; window=N_short);
    global ema_long = compute_ema(market_prices; window=N_long);
    global lambda_series = compute_lambda(ema_short, ema_long; G=GAIN);
    lambda_series[1:offset] .= 0.0;

    # Compute market growth and smooth with EMA -
    gm_raw = compute_market_growth(market_prices; Δt=Δt);
    global gm_ema = compute_ema(gm_raw; window=N_growth);

    # Generate per-ticker prices via SIM -
    global price_matrix = zeros(T_total, K + 1);
    price_matrix[:, 1] = 1:T_total;

    start_prices = Dict("LargeCap" => 150.0, "SmallCap" => 45.0,
        "International" => 80.0, "Bond" => 100.0, "Commodity" => 60.0);

    for (k, ticker) in enumerate(tickers)
        (αᵢ, βᵢ, σᵢ) = sim_params[ticker];
        price_matrix[1, k + 1] = start_prices[ticker];
        for t in 2:T_total
            gm_t = gm_raw[min(t - 1, length(gm_raw))];
            gᵢ = αᵢ + βᵢ * gm_t * Δt + σᵢ * sqrt(Δt) * randn();
            price_matrix[t, k + 1] = price_matrix[t - 1, k + 1] * exp(gᵢ);
        end
    end

    println("Generated $(T_total) days of synthetic market data")
    println("Lambda range: [$(round(minimum(lambda_series), digits=3)), $(round(maximum(lambda_series), digits=3))]")
end

**Visualize:** Market price with EMAs (top) and the lambda sentiment signal (bottom).

In [ ]:
let
    T_total = length(market_prices);
    days = 1:T_total;

    # Top: price + EMAs -
    p1 = plot(days, market_prices, label="Price", linewidth=1.5, color=:gray70, alpha=0.8)
    plot!(p1, days, ema_short, label="EMA-$(N_short)", linewidth=2, color=:steelblue)
    plot!(p1, days, ema_long, label="EMA-$(N_long)", linewidth=2, color=:coral)
    vline!(p1, [offset], label="Warmup end", linestyle=:dash, color=:black, alpha=0.5)
    ylabel!(p1, "Price (\$)")
    title!(p1, "Synthetic Market Index with EMAs")

    # Bottom: lambda signal -
    p2 = plot(days, lambda_series, label="λ (sentiment)", linewidth=2, color=:purple)
    hline!(p2, [0.0], label="", linestyle=:dash, color=:black, alpha=0.3)
    vline!(p2, [offset], label="Warmup end", linestyle=:dash, color=:black, alpha=0.5)
    xlabel!(p2, "Trading Day")
    ylabel!(p2, "λ")
    title!(p2, "Sentiment Signal (λ > 0 = bearish, λ < 0 = bullish)")

    plot(p1, p2, layout=(2, 1), size=(800, 550), legend=:topright)
end

___
## Task 2: Cobb-Douglas Utility Allocation
Now we use the Cobb-Douglas utility function to allocate capital. At each time step, we compute preference weights $\gamma_i$ from SIM parameters and the current sentiment $\lambda_t$, then solve the analytical allocation:

$$n_i^{\star} = \frac{\gamma_i}{\sum_{j \in S^+} \gamma_j} \cdot \frac{B_{\text{adj}}}{P_i} \quad \text{(preferred)}, \qquad n_i^{\star} = \epsilon \quad \text{(non-preferred)}$$

> **What should you see?** Preference weights shift over time as lambda changes. During bearish periods, high-beta assets (SmallCap) lose preference; during bullish periods, they gain. The share allocation responds accordingly — budget flows toward preferred assets.

In [ ]:
let
    K = length(tickers);
    T_trade = n_trading_days;
    ε = 0.1;  # minimum share floor

    # Storage for time series -
    global gamma_matrix = zeros(T_trade, K);  # preferences over time
    global shares_matrix = zeros(T_trade, K); # shares over time
    global utility_series = zeros(T_trade);    # CD utility values

    for d in 1:T_trade
        actual_day = offset + d;
        λ_t = lambda_series[actual_day];
        gm_t = gm_ema[min(actual_day, length(gm_ema))];
        prices = [price_matrix[actual_day, k + 1] for k in 1:K];

        # Compute preference weights from SIM + sentiment -
        gamma = compute_preference_weights(sim_params, tickers, gm_t, λ_t);

        # Solve Cobb-Douglas allocation -
        problem = build(MyCobbDouglasChoiceProblem, (
            gamma = gamma, prices = prices, B = B₀, epsilon = ε
        ));
        (shares, cash) = allocate_cobb_douglas(problem);

        # Store -
        gamma_matrix[d, :] = gamma;
        shares_matrix[d, :] = shares;
        utility_series[d] = evaluate_cobb_douglas(shares, gamma);
    end

    println("Cobb-Douglas allocation computed for $(T_trade) trading days")
    println("Mean utility: $(round(mean(utility_series), digits=2))")
end

**Visualize:** Preference weights $\gamma_i$ over time (top) and corresponding share allocations (bottom).

> **What should you see?** The top panel shows how each asset's preference weight moves with sentiment. Bond ($\beta = -0.15$) should be relatively stable since its low beta makes it less sensitive to $\lambda$. SmallCap ($\beta = 1.35$) should swing the most. The bottom panel shows the resulting share counts — budget flows to whichever assets have the highest $\gamma_i$.

In [ ]:
let
    days = 1:n_trading_days;
    colors = [:steelblue :coral :green :goldenrod :purple];

    # Top: preference weights -
    p1 = plot(size=(800, 250), title="Cobb-Douglas Preference Weights γᵢ Over Time",
        ylabel="γᵢ", legend=:outerright)
    for (k, ticker) in enumerate(tickers)
        plot!(p1, days, gamma_matrix[:, k], label=ticker, linewidth=1.5, color=colors[k])
    end
    hline!(p1, [0.0], label="", linestyle=:dash, color=:black, alpha=0.3)

    # Bottom: share allocations -
    p2 = plot(size=(800, 250), title="Cobb-Douglas Share Allocation nᵢ* Over Time",
        xlabel="Trading Day", ylabel="Shares", legend=:outerright)
    for (k, ticker) in enumerate(tickers)
        plot!(p2, days, shares_matrix[:, k], label=ticker, linewidth=1.5, color=colors[k])
    end

    plot(p1, p2, layout=(2, 1), size=(800, 550))
end

___
## Task 3: Compare Utility Functions — Cobb-Douglas vs. CES vs. Log-Linear
We run the same scenario with three different utility functions and compare allocation behavior. The key question: **how does the choice of utility function affect portfolio concentration?**

CES with high $\sigma$ (elasticity of substitution) produces more concentrated portfolios — it pushes more budget toward the single best asset. Cobb-Douglas is the middle ground. Log-linear produces the same allocation as Cobb-Douglas but different utility values.

> **What should you see?** Cobb-Douglas and log-linear allocations will be identical (the log transform preserves the optimum). CES with $\sigma = 3$ will be more concentrated — the top asset gets a larger share of the budget. CES with $\sigma = 0.5$ will be more diversified.

In [ ]:
let
    K = length(tickers);
    T_trade = n_trading_days;
    ε = 0.1;

    # Storage: shares for each utility function -
    shares_cd = zeros(T_trade, K);       # Cobb-Douglas
    shares_ces_high = zeros(T_trade, K); # CES σ = 3.0 (concentrated)
    shares_ces_low = zeros(T_trade, K);  # CES σ = 0.5 (diversified)
    shares_ll = zeros(T_trade, K);       # Log-linear

    # Utility values -
    utility_cd = zeros(T_trade);
    utility_ces_high = zeros(T_trade);
    utility_ces_low = zeros(T_trade);
    utility_ll = zeros(T_trade);

    for d in 1:T_trade
        actual_day = offset + d;
        λ_t = lambda_series[actual_day];
        gm_t = gm_ema[min(actual_day, length(gm_ema))];
        prices = [price_matrix[actual_day, k + 1] for k in 1:K];

        gamma = compute_preference_weights(sim_params, tickers, gm_t, λ_t);

        # Cobb-Douglas -
        cd_prob = build(MyCobbDouglasChoiceProblem, (gamma=gamma, prices=prices, B=B₀, epsilon=ε));
        (s_cd, _) = allocate_cobb_douglas(cd_prob);
        shares_cd[d, :] = s_cd;
        utility_cd[d] = evaluate_cobb_douglas(s_cd, gamma);

        # CES high σ (concentrated) -
        ces_h = build(MyCESChoiceProblem, (gamma=gamma, prices=prices, B=B₀, epsilon=ε, sigma=3.0));
        (s_ces_h, _) = allocate_ces(ces_h);
        shares_ces_high[d, :] = s_ces_h;
        utility_ces_high[d] = evaluate_ces(s_ces_h, gamma; sigma=3.0);

        # CES low σ (diversified) -
        ces_l = build(MyCESChoiceProblem, (gamma=gamma, prices=prices, B=B₀, epsilon=ε, sigma=0.5));
        (s_ces_l, _) = allocate_ces(ces_l);
        shares_ces_low[d, :] = s_ces_l;
        utility_ces_low[d] = evaluate_ces(s_ces_l, gamma; sigma=0.5);

        # Log-linear -
        ll_prob = build(MyLogLinearChoiceProblem, (gamma=gamma, prices=prices, B=B₀, epsilon=ε));
        (s_ll, _) = allocate_log_linear(ll_prob);
        shares_ll[d, :] = s_ll;
        utility_ll[d] = evaluate_log_linear(s_ll, gamma);
    end

    # Store for use below -
    global _shares_cd = shares_cd;
    global _shares_ces_high = shares_ces_high;
    global _shares_ces_low = shares_ces_low;
    global _shares_ll = shares_ll;
    global _utility_cd = utility_cd;
    global _utility_ces_high = utility_ces_high;
    global _utility_ces_low = utility_ces_low;
    global _utility_ll = utility_ll;

    println("Allocation computed for all 4 utility variants")
end

**Visualize:** Concentration comparison — what fraction of budget goes to the top asset under each utility function?

> **What should you see?** CES ($\sigma = 3$) should have the highest concentration — more budget in the top asset. Cobb-Douglas and log-linear should be identical. CES ($\sigma = 0.5$) should be the most diversified.

In [ ]:
let
    K = length(tickers);
    T_trade = n_trading_days;

    # Compute top-asset budget fraction for each utility function -
    function top_asset_fraction(shares_mat)
        fracs = zeros(T_trade);
        for d in 1:T_trade
            actual_day = offset + d;
            values = [shares_mat[d, k] * price_matrix[actual_day, k + 1] for k in 1:K];
            fracs[d] = maximum(values) / sum(values);
        end
        return fracs;
    end

    f_cd = top_asset_fraction(_shares_cd);
    f_ces_h = top_asset_fraction(_shares_ces_high);
    f_ces_l = top_asset_fraction(_shares_ces_low);
    f_ll = top_asset_fraction(_shares_ll);

    days = 1:T_trade;
    plot(days, f_cd, label="Cobb-Douglas", linewidth=2, color=:steelblue)
    plot!(days, f_ces_h, label="CES (σ=3, concentrated)", linewidth=2, color=:coral)
    plot!(days, f_ces_l, label="CES (σ=0.5, diversified)", linewidth=2, color=:green)
    plot!(days, f_ll, label="Log-Linear", linewidth=2, color=:purple, linestyle=:dash)
    xlabel!("Trading Day")
    ylabel!("Top-Asset Budget Fraction")
    title!("Portfolio Concentration by Utility Function")
    plot!(size=(800, 400), legend=:topright)
end

**Compare:** Summary statistics across utility functions.

In [ ]:
let
    K = length(tickers);
    T_trade = n_trading_days;

    function concentration_stats(shares_mat)
        hhi_vals = zeros(T_trade); # Herfindahl-Hirschman Index
        for d in 1:T_trade
            actual_day = offset + d;
            values = [shares_mat[d, k] * price_matrix[actual_day, k + 1] for k in 1:K];
            weights = values ./ sum(values);
            hhi_vals[d] = sum(weights .^ 2);
        end
        return (mean=round(mean(hhi_vals), digits=4), std=round(std(hhi_vals), digits=4))
    end

    names = ["Cobb-Douglas", "CES (σ=3.0)", "CES (σ=0.5)", "Log-Linear"];
    mats = [_shares_cd, _shares_ces_high, _shares_ces_low, _shares_ll];
    stats = [concentration_stats(m) for m in mats];

    data = hcat(names,
        [s.mean for s in stats],
        [s.std for s in stats]);

    pretty_table(data,
        header=["Utility Function", "Mean HHI", "Std HHI"],
        alignment=[:l, :c, :c])
end

___
## Summary
In this example, we built the complete Cobb-Douglas utility allocation pipeline: synthetic market data → SIM parameters + EMA sentiment → preference weights $\gamma_i$ → analytical allocation. We compared four utility variants and showed how the elasticity of substitution controls portfolio concentration.

Key observations:
* Cobb-Douglas and log-linear produce **identical allocations** (the log transform preserves the optimum)
* CES with high $\sigma$ **concentrates** the portfolio — useful when you have high conviction
* CES with low $\sigma$ **diversifies** — useful when you want robustness
* All utility functions use the same preference weights $\gamma_i$ — the difference is in how they translate preferences into shares

In the next example, we wire the Cobb-Douglas allocator into the full rebalancing engine and produce a formal scorecard.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice. The examples use synthetic data and simplified models.